In [1]:
#!/usr/bin/env python3
import re
import csv
from itertools import combinations
from pathlib import Path

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize


# =========================
# CONFIG
# =========================
INPUT_FILE = r"C:/Users/ak4086/Desktop/Data TEMET/sir_ref.txt"
OUTPUT_FOLDER = r"C:/Users/ak4086/Desktop/Data TEMET/output"

TITLE = "SIR"

LAYOUT_SEED = 42
EDGE_ALPHA = 0.20
EDGE_WIDTH = 0.6

NODE_BASE_SIZE = 180
NODE_SIZE_GAIN = 520

TITLE_FONTSIZE = 32
TITLE_FONTWEIGHT = "bold"
TITLE_PAD = 25
COLORBAR_LABELSIZE = 15


# =========================
# HELPERS
# =========================
def clean_text(s):
    return re.sub(r"\s+", " ", (s or "").strip())


def norm_ref(s):
    s = (s or "").lower().strip()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# =========================
# PARSE WOS PLAINTEXT
# =========================
def parse_wos_plaintext(file_path):
    papers = {}

    current_ut = None
    authors = []
    refs = []

    in_authors = False
    in_refs = False

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.rstrip("\n")

            if line.startswith("ER"):
                if current_ut:
                    papers[current_ut] = {
                        "authors": list(dict.fromkeys(authors)),
                        "refs": list(dict.fromkeys(refs)),
                    }

                current_ut = None
                authors = []
                refs = []
                in_authors = False
                in_refs = False
                continue

            if line.startswith("UT"):
                current_ut = clean_text(line[2:])
                in_authors = False
                in_refs = False
                continue

            if line.startswith("AU"):
                in_authors = True
                in_refs = False
                author = clean_text(line[2:])
                if author:
                    authors.append(author)
                continue

            if in_authors and (line.startswith(" ") or line.startswith("\t")):
                author = clean_text(line)
                if author:
                    authors.append(author)
                continue

            if line.startswith("CR"):
                in_refs = True
                in_authors = False
                ref = norm_ref(line[2:])
                if ref:
                    refs.append(ref)
                continue

            if in_refs and (line.startswith(" ") or line.startswith("\t")):
                ref = norm_ref(line)
                if ref:
                    refs.append(ref)
                continue

            if re.match(r"^[A-Z]{2}\b", line[:2]):
                in_authors = False
                in_refs = False

    return papers


# =========================
# BUILD EDGE LISTS
# =========================
def build_coref_edges(papers):
    edges = []
    ids = list(papers.keys())

    for p1, p2 in combinations(ids, 2):
        r1 = set(papers[p1]["refs"])
        r2 = set(papers[p2]["refs"])

        if not r1 or not r2:
            continue

        shared = r1.intersection(r2)

        if shared:
            shared_count = len(shared)
            weight = shared_count / min(len(r1), len(r2))

            edges.append([
                p1,
                p2,
                shared_count,
                round(weight, 6),
                "shared_references"
            ])

    return edges


def build_coauthor_edges(papers):
    edge_weights = {}

    for paper_id, data in papers.items():
        authors = data["authors"]

        if len(authors) < 2:
            continue

        for a1, a2 in combinations(sorted(authors), 2):
            key = tuple(sorted([a1, a2]))
            edge_weights[key] = edge_weights.get(key, 0) + 1

    edges = []

    for (a1, a2), weight in edge_weights.items():
        edges.append([
            a1,
            a2,
            weight,
            "coauthorship"
        ])

    return edges


# =========================
# SAVE EDGE LISTS
# =========================
def save_coref_edges(edges, output_file):
    with open(output_file, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["source", "target", "shared_count", "weight", "edge_type"])
        writer.writerows(edges)


def save_coauthor_edges(edges, output_file):
    with open(output_file, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["source", "target", "weight", "edge_type"])
        writer.writerows(edges)


# =========================
# MAKE NETWORKS
# =========================
def make_coref_graph(coref_edges):
    G = nx.Graph()

    for source, target, shared_count, weight, edge_type in coref_edges:
        G.add_edge(
            source,
            target,
            shared_count=shared_count,
            weight=weight,
            edge_type=edge_type
        )

    return G


def make_coauthor_graph(coauthor_edges):
    G = nx.Graph()

    for source, target, weight, edge_type in coauthor_edges:
        G.add_edge(
            source,
            target,
            weight=weight,
            edge_type=edge_type
        )

    return G


# =========================
# SCORES FOR NODE SIZE/COLOR
# =========================
def coref_scores(G):
    scores = {}

    for n in G.nodes():
        scores[n] = sum(
            d.get("shared_count", 0)
            for _, _, d in G.edges(n, data=True)
        )

    return scores


def coauthor_scores(G):
    scores = {}

    for n in G.nodes():
        scores[n] = sum(
            d.get("weight", 0)
            for _, _, d in G.edges(n, data=True)
        )

    return scores


# =========================
# PLOT NETWORK
# =========================
def plot_network(G, scores, out_png, title, colorbar_label):
    if G.number_of_nodes() == 0:
        print(f"Empty graph, skipping plot: {title}")
        return

    pos = nx.spring_layout(G, seed=LAYOUT_SEED)

    vals = list(scores.values())

    vmin = min(vals)
    vmax = max(vals)

    if vmax == vmin:
        vmax = vmin + 1e-9

    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.get_cmap("Reds")

    node_list = list(G.nodes())
    node_scores = [scores.get(n, 0) for n in node_list]

    node_colors = [cmap(norm(s)) for s in node_scores]

    scaled = [
        (s - vmin) / (vmax - vmin)
        for s in node_scores
    ]

    node_sizes = [
        NODE_BASE_SIZE + NODE_SIZE_GAIN * x
        for x in scaled
    ]

    plt.figure(figsize=(14, 10), dpi=300)
    ax = plt.gca()

    ax.set_title(
        title,
        fontsize=TITLE_FONTSIZE,
        fontweight=TITLE_FONTWEIGHT,
        pad=TITLE_PAD
    )

    nx.draw_networkx_edges(
        G,
        pos,
        alpha=EDGE_ALPHA,
        width=EDGE_WIDTH
    )

    nx.draw_networkx_nodes(
        G,
        pos,
        node_color=node_colors,
        node_size=node_sizes,
        edgecolors="black",
        linewidths=0.3
    )

    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(colorbar_label, fontsize=COLORBAR_LABELSIZE)

    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()


# =========================
# BASIC NETWORK STATS
# =========================
def print_stats(G, name):
    print(f"\n{name}")
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())

    if G.number_of_nodes() > 1:
        print("Density:", round(nx.density(G), 6))
        print("Connected components:", nx.number_connected_components(G))
        print("Average clustering:", round(nx.average_clustering(G), 6))


# =========================
# MAIN
# =========================
def main():
    input_path = Path(INPUT_FILE)
    output_path = Path(OUTPUT_FOLDER)
    output_path.mkdir(parents=True, exist_ok=True)

    coref_csv = output_path / "sir_ref_coref_edgelist.csv"
    coauthor_csv = output_path / "sir_ref_coauthor_edgelist.csv"

    coref_png = output_path / "sir_ref_coref_network.png"
    coauthor_png = output_path / "sir_ref_coauthor_network.png"

    papers = parse_wos_plaintext(input_path)

    print("Papers parsed:", len(papers))

    coref_edges = build_coref_edges(papers)
    coauthor_edges = build_coauthor_edges(papers)

    save_coref_edges(coref_edges, coref_csv)
    save_coauthor_edges(coauthor_edges, coauthor_csv)

    print("Coreference edges:", len(coref_edges))
    print("Coauthorship edges:", len(coauthor_edges))

    G_coref = make_coref_graph(coref_edges)
    G_coauthor = make_coauthor_graph(coauthor_edges)

    scores_coref = coref_scores(G_coref)
    scores_coauthor = coauthor_scores(G_coauthor)

    plot_network(
        G_coref,
        scores_coref,
        coref_png,
        TITLE,
        "Node heat = total shared references"
    )

    plot_network(
        G_coauthor,
        scores_coauthor,
        coauthor_png,
        "Coauthorship Network",
        "Node heat = collaboration strength"
    )

    print_stats(G_coref, "Coreference network")
    print_stats(G_coauthor, "Coauthorship network")

    print("\nSaved files:")
    print(coref_csv)
    print(coauthor_csv)
    print(coref_png)
    print(coauthor_png)


if __name__ == "__main__":
    main()

Papers parsed: 53
Coreference edges: 212
Coauthorship edges: 271

Coreference network
Nodes: 47
Edges: 212
Density: 0.196115
Connected components: 1
Average clustering: 0.604662

Coauthorship network
Nodes: 138
Edges: 271
Density: 0.028668
Connected components: 31
Average clustering: 0.77936

Saved files:
C:\Users\ak4086\Desktop\Data TEMET\output\sir_ref_coref_edgelist.csv
C:\Users\ak4086\Desktop\Data TEMET\output\sir_ref_coauthor_edgelist.csv
C:\Users\ak4086\Desktop\Data TEMET\output\sir_ref_coref_network.png
C:\Users\ak4086\Desktop\Data TEMET\output\sir_ref_coauthor_network.png
